In [2]:
import requests
import json
import pandas as pd

/Users/dylan/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/dylan/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [64]:
# this api has predictions and odds in it
api_football_token = '3b7345f58043e45e5d42c076511e08e7'
api_football_api = 'https://v3.football.api-sports.io/'

{'get': 'teams',
 'parameters': {'league': '39', 'season': '2023'},
 'errors': [],
 'results': 20,
 'paging': {'current': 1, 'total': 1},
 'response': [{'team': {'id': 33,
    'name': 'Manchester United',
    'code': 'MUN',
    'country': 'England',
    'founded': 1878,
    'national': False,
    'logo': 'https://media.api-sports.io/football/teams/33.png'},
   'venue': {'id': 556,
    'name': 'Old Trafford',
    'address': 'Sir Matt Busby Way',
    'city': 'Manchester',
    'capacity': 76212,
    'surface': 'grass',
    'image': 'https://media.api-sports.io/football/venues/556.png'}},
  {'team': {'id': 34,
    'name': 'Newcastle',
    'code': 'NEW',
    'country': 'England',
    'founded': 1892,
    'national': False,
    'logo': 'https://media.api-sports.io/football/teams/34.png'},
   'venue': {'id': 562,
    'name': "St. James' Park",
    'address': 'St. James&apos; Street',
    'city': 'Newcastle upon Tyne',
    'capacity': 52758,
    'surface': 'grass',
    'image': 'https://media.

In [118]:
# get teams
r = requests.get(api_football_api+'/teams?league=39&season=2023', headers={'x-rapidapi-key':api_football_token})
teams_response = json.loads(r.text)['response']

In [119]:
venues = dict()
teams = dict()
for team in teams_response:
    venues[team['venue'].pop('id')] = team['venue']
    teams[team['team'].pop('id')] = team['team']

In [122]:
# get players
paged_
r = requests.get(api_football_api+'/players?league=39&season=2023', headers={'x-rapidapi-key':api_football_token})
players_response = json.loads(r.text)['response']

In [145]:
players = dict()
players_url = api_football_api+"/players?league=39&season=2023&page={}"
page_number = 1
while True:
    r = requests.get(players_url.format(page_number), headers={'x-rapidapi-key':api_football_token})
    players_response = json.loads(r.text)
    
    for player in players_response['response']:
        del player['statistics']
        players[player['player'].pop('id')] = player['player']

    if players_response['paging']['current'] == players_response['paging']['total']:
        break
    page_number+=1

In [148]:
players

{54: {'name': 'Diego Costa',
  'firstname': 'Diego',
  'lastname': 'da Silva Costa',
  'age': 37,
  'birth': {'date': '1988-10-07', 'place': 'Lagarto', 'country': 'Brazil'},
  'nationality': 'Spain',
  'height': '188 cm',
  'weight': '83 kg',
  'injured': False,
  'photo': 'https://media.api-sports.io/football/players/54.png'},
 178: {'name': 'Lucas Moura',
  'firstname': 'Lucas',
  'lastname': 'Rodrigues Moura da Silva',
  'age': 33,
  'birth': {'date': '1992-08-13', 'place': 'São Paulo', 'country': 'Brazil'},
  'nationality': 'Brazil',
  'height': '172 cm',
  'weight': '72 kg',
  'injured': False,
  'photo': 'https://media.api-sports.io/football/players/178.png'},
 297: {'name': 'A. Oxlade-Chamberlain',
  'firstname': 'Alexander Mark David',
  'lastname': 'Oxlade-Chamberlain',
  'age': 32,
  'birth': {'date': '1993-08-15', 'place': 'Portsmouth', 'country': 'England'},
  'nationality': 'England',
  'height': '180 cm',
  'weight': '70 kg',
  'injured': False,
  'photo': 'https://media.

In [72]:
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable
import logging
import pandas as pd

URI = "neo4j+s://d84d3aca.databases.neo4j.io"
user = "neo4j"
password = '5fX_nXMa0jDLsTdhEz2cbNKf46sZn4Z8Fg7O4MLUQek'
AUTH = (user, password)
driver = GraphDatabase.driver(URI, auth=AUTH)

In [128]:
query_player = """
MERGE (p:Player {id: $id})
SET p.name = $name, 
    p.dateOfBirth = $dateOfBirth,
    p.nationality = $nationality
RETURN p
"""

query_position = """
MERGE (pos:Position {name: $position})
RETURN pos
"""

query_relationship = """
MATCH (p:Player {id: $id}), (pos:Position {name: $position})
MERGE (p)-[:PLAYS_AS]->(pos)
"""

# Example list of players


with driver.session(database="neo4j") as session:
    for player in players:
        # Create Player node
        session.run(
            query_player,
            id=player['id'],
            name=player['name'],
            dateOfBirth=player['dateOfBirth'],
            nationality=player['nationality']
        )
        
        # Create Position node
        session.run(
            query_position,
            position=player['position']
        )
        
        # Create relationship between Player and Position
        session.run(
            query_relationship,
            id=player['id'],
            position=player['position']
        )


In [149]:
coaches = [ team['coach']for team in data['teams']]

In [150]:
coaches[0]

{'id': 179744,
 'firstName': '',
 'lastName': 'Mikel Arteta',
 'name': 'Mikel Arteta',
 'dateOfBirth': '1982-03-26',
 'nationality': 'Spain',
 'contract': {'start': '2019-12', 'until': '2027-06'}}

In [144]:
players_id = [p['id'] for p in players]

In [159]:
data['teams'][0]

{'area': {'id': 2072,
  'name': 'England',
  'code': 'ENG',
  'flag': 'https://crests.football-data.org/770.svg'},
 'id': 57,
 'name': 'Arsenal FC',
 'shortName': 'Arsenal',
 'tla': 'ARS',
 'crest': 'https://crests.football-data.org/57.png',
 'address': '75 Drayton Park London N5 1BU',
 'website': 'http://www.arsenal.com',
 'founded': 1886,
 'clubColors': 'Red / White',
 'venue': 'Emirates Stadium',
 'runningCompetitions': [{'id': 2021,
   'name': 'Premier League',
   'code': 'PL',
   'type': 'LEAGUE',
   'emblem': 'https://crests.football-data.org/PL.png'},
  {'id': 2001,
   'name': 'UEFA Champions League',
   'code': 'CL',
   'type': 'CUP',
   'emblem': 'https://crests.football-data.org/CL.png'}],
 'coach': {'id': 179744,
  'firstName': '',
  'lastName': 'Mikel Arteta',
  'name': 'Mikel Arteta',
  'dateOfBirth': '1982-03-26',
  'nationality': 'Spain',
  'contract': {'start': '2019-12', 'until': '2027-06'}},
 'squad': [{'id': 3221,
   'name': 'Neto',
   'position': 'Goalkeeper',
   'd